### Задача №4 

#### Пункт 1

In [ ]:
WITH создаем сте для получения нужной информации

    -- получаем дату начала текущего месяца, высчитываем дату начала прошлого месяца и дату 3 месяца назад 
    toStartOfMonth(today())                AS cur_m_start,    
    cur_m_start - INTERVAL 1 MONTH         AS last_m_start,   
    cur_m_start - INTERVAL 3 MONTH         AS three_m_start,  

    -- выручка в кол-ве и в рублях за прошлый полный месяц
    sale_month AS (
        SELECT
            itemid,
            sum(qty)                         AS sales_qty,
            sum(qty * price)                 AS sales_rub,
            sum(qty * price) / sum(qty)      AS avg_price
        FROM dwh.sales
        WHERE date >= last_m_start
          AND date <  cur_m_start
        GROUP BY itemid
        HAVING sum(qty) > 0
    ),

    -- выручка в кол-ве и в рублях за 3 месяца, включая полный последний
    sale_three_months AS (
        SELECT
            itemid,
            sum(qty) / 3 AS avg_qty_3m
        FROM dwh.sales
        WHERE date >= three_m_start
          AND date <  cur_m_start
        GROUP BY itemid
    ),

    -- текущий остаток, остаток на максимальную дату  
    stock AS (
        SELECT
            itemid,
            sum(qty) AS stock_qty
        FROM dwh.stock
        WHERE date = (SELECT max(date) FROM dwh.stock)
        GROUP BY itemid
    )

SELECT
    pr.itemid AS "код_товара",
    pr.name_ax AS "название_товара",
    pr.brand_name_ax AS "бренд",
    pr.category AS "категория",
    sm.sales_qty AS "выручка_месяц, шт",
    sm.sales_rub AS "выручка_месяц, руб",
    round(sm.avg_price, 2) AS "средняя_цена",
    round(stm.avg_qty_3m, 2) AS "средние_продажи_3мес, шт",
    cs.stock_qty AS "остаток, шт",
    round(cs.stock_qty - stm.avg_qty_3m, 2) AS "остаток_минус_средние_продажи"

FROM dwh.products AS pr

INNER JOIN sale_month    AS sm ON sm.itemid = pr.itemid   -- фильтруем те, продажи по которым были за последний месяц
LEFT  JOIN sale_three_months  AS stm ON stm.itemid = pr.itemid
LEFT  JOIN stock AS cs ON cs.itemid = pr.itemid

#### Пункт 2

In [3]:
WITH -- создаем сте для получения нужной информации
    toStartOfMonth(today())          AS cur_m,
    cur_m_start - INTERVAL 1 MONTH   AS last_m,
    cur_m_start - INTERVAL 2 MONTH   AS prev_m,

    -- продажи брендов за last_m
    last_month AS (
        SELECT
            p.brand_name_ax AS brand,
            sum(s.qty * s.price) AS sales_rub
        FROM dwh.sales AS s
        INNER JOIN dwh.products AS p ON p.itemid = s.itemid
        WHERE s.date >= last_m
          AND s.date <  cur_m
        GROUP BY p.brand_name_ax
    ),

    -- продажи брендов за prev_m
    prev_month AS (
        SELECT
            p.brand_name_ax AS brand,
            sum(s.qty * s.price) AS sales_rub
        FROM dwh.sales AS s
        INNER JOIN dwh.products AS p ON p.itemid = s.itemid
        WHERE s.date >= prev_m
          AND s.date <  last_m
        GROUP BY p.brand_name_ax
    )

SELECT
    lm.brand AS "бренд",
    lm.sales_rub AS "прошлый_месяц, руб",
    pm.sales_rub AS "позапрошлый_месяц, руб",
    lm.sales_rub - pm.sales_rub AS "diff, руб"

FROM last_month AS lm
LEFT JOIN prev_month AS pm ON pm.brand = lm.brand

-- сортировка и отбор топ 5
ORDER BY lm.sales_rub DESC
LIMIT 5

IndentationError: unexpected indent (1989326255.py, line 2)